# Análisis complementario: NARDL de precios (asimetría de alzas y bajas)

José Andrés Estrada Morales · Carné 1031722

Responde a la observación del Dr. Herrera (comentario a las hipótesis y reunión de retroalimentación):
la volatilidad mide la magnitud de las fluctuaciones, no su dirección. Este análisis separa las alzas y
las bajas de cada precio para ver si la dirección importa. Es **complementario**: no cambia el título,
los objetivos ni las hipótesis, que siguen centrados en la volatilidad (checklist C09, opción B).

Método (Shin, Yu y Greenwood-Nimmo, 2014): con x_t = 100 × ln(P_t),
POS_t = Σ máx(Δx_j, 0) y NEG_t = Σ mín(Δx_j, 0). Se estima un NARDL por producto con POS, NEG y los
mismos controles (ITCER, TCN, remesas), 11 dummies mensuales, rezagos por BIC y las mismas pruebas del
código v1: Bounds F, t y Fx con valores críticos de muestra finita, errores convencionales, HC3 y HAC.

Asimetría de largo plazo: H0: θ⁺ = θ⁻ (coeficientes en niveles de POS y NEG).
Asimetría de corto plazo: H0: Σ ΔPOS = Σ ΔNEG. Holm entre los cinco productos.

Interpretación de los multiplicadores: L⁺ es el cambio de largo plazo del saldo ante un alza acumulada de 1% del
precio; ante una baja acumulada de 1%, el saldo cambia en −L⁻ (NEG baja cuando el precio baja).

Requiere `codigo_tesis_v1.py` y `Base_datos_tesis.xlsx` en la misma carpeta.

In [ ]:
ARCHIVO = 'Base_datos_tesis.xlsx'      # base en la misma carpeta que este archivo
SALIDA = 'resultados_nardl_saldo_normalizado'            # carpeta de resultados
DEPENDIENTE = 'saldo_norm'                               # 'saldo_norm' (normalizado) o 'balanza' (US$ millones)
NSIM = 3000                                              # simulaciones para valores críticos de muestra finita

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.multitest import multipletests

import codigo_tesis_v1 as v1


def sumas_parciales(df):
    """Descomposición en sumas parciales de 100 × ln(precio) para cada producto."""
    for c in v1.PRECIOS:
        dx = 100 * np.log(df[c]).diff()
        df[f'{c}_pos'] = dx.clip(lower=0).fillna(0).cumsum()
        df[f'{c}_neg'] = dx.clip(upper=0).fillna(0).cumsum()
        df[f'lnp_{c}'] = 100 * np.log(df[c])
    return df


def _wald(res, filas):
    """Wald para H0: Σ a·β = 0, con filas = lista de (término, peso)."""
    R = np.zeros((1, len(res.params)))
    for t, w in filas:
        R[0, res.params.index.get_loc(t)] += w
    w = res.wald_test(R, scalar=True)
    return float(w.statistic), float(w.pvalue)


def nardl_producto(ad, y, c, S, salida, nsim):
    xcols = [f'{c}_pos', f'{c}_neg'] + v1.CONTROLES
    fila, conv, D, dy, (p, q) = v1.estimar(ad, y, xcols, S, 'BIC', f'nardl_{c}', salida, vols=[],
                                           nsim=nsim, completo=True)
    n = int(conv.nobs)
    res = {'conv': conv,
           'HC3': OLS(dy, D).fit(cov_type='HC3'),
           'HAC': OLS(dy, D).fit(cov_type='HAC', cov_kwds={'maxlags': v1.hac_lags(n)})}
    pos, neg = f'{c}_pos', f'{c}_neg'
    corto = ([(t, 1) for t in D if t.startswith(f'D.{pos}.L')] +
             [(t, -1) for t in D if t.startswith(f'D.{neg}.L')])
    out = dict(producto=v1.NOMBRES[c], p=p, q=fila['q'], n=n,
               F=fila['F'], F_I1_5=fila['F_I1_5'], t=fila['t'], t_I1_5=fila['t_I1_5'],
               Fx=fila['Fx'], Fx_I1_5=fila['Fx_I1_5'], relacion_en_niveles=fila['relacion_en_niveles'],
               coef_saldo_L1=fila['coef_saldo_L1'], BG12_p=fila['BG12_p'], BG24_p=fila['BG24_p'],
               BP_p=fila['BP_p'], JB_p=fila['JB_p'])
    for cov, r in res.items():
        out[f'asimetria_largo_{cov}_p'] = _wald(r, [(f'{pos}.L1', 1), (f'{neg}.L1', -1)])[1]
        out[f'asimetria_corto_{cov}_p'] = _wald(r, corto)[1]
    m = pd.read_csv(salida / f'05_multiplicadores_nardl_{c}.csv').set_index('variable')
    for signo, col in [('pos', pos), ('neg', neg)]:
        out[f'L_{signo}'] = float(m.loc[f'{col}.L1', 'multiplicador'])
        out[f'L_{signo}_ee'] = float(m.loc[f'{col}.L1', 'ee_delta'])
        out[f'L_{signo}_p'] = float(m.loc[f'{col}.L1', 'p_aprox'])
    return out


def figura(t, salida, y):
    fig, ax = plt.subplots(figsize=(10, 4.8))
    x = np.arange(len(t))
    for j, (signo, etiqueta, color) in enumerate([('pos', 'Alzas (L⁺)', '#1F4E79'), ('neg', 'Bajas (L⁻)', '#E36C09')]):
        ax.errorbar(x + (j - .5) * .25, t[f'L_{signo}'], yerr=1.96 * t[f'L_{signo}_ee'], fmt='o', capsize=4,
                    label=etiqueta, color=color)
    ax.axhline(0, color='k', lw=.7)
    ax.set_xticks(x, t.producto)
    unidad = 'pp del saldo normalizado' if y == 'saldo_norm' else 'US$ millones'
    ax.set_ylabel(f'Multiplicador de largo plazo\n({unidad} por 1% de cambio acumulado)')
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, hi + 0.35 * (hi - lo))
    for i, r in t.reset_index(drop=True).iterrows():
        ax.text(i, 0.97, f"Asimetría de largo plazo\np Holm (HAC) = {r['asimetria_largo_HAC_p_Holm']:.3f}",
                ha='center', va='top', fontsize=8, transform=ax.get_xaxis_transform())
    ax.set_title('NARDL de precios: multiplicadores de largo plazo de alzas y bajas (IC 95%, método delta)', pad=10)
    ax.legend(loc='lower right')
    fig.tight_layout(rect=[0, .05, 1, 1])
    v1._nota(fig, f'{v1.FUENTE_COMERCIO}; {v1.FUENTE_PRECIOS}; SECMCA')
    fig.savefig(salida / 'fig_nardl_multiplicadores.png', dpi=200)
    plt.close(fig)
    return fig


def main(archivo=ARCHIVO, salida=SALIDA, y=DEPENDIENTE, nsim=NSIM):
    salida = Path(salida)
    salida.mkdir(parents=True, exist_ok=True)
    df, _ = v1.cargar_datos(Path(archivo))
    df = sumas_parciales(df)
    # Orden de integración de precios y sumas parciales: se descarta I(2) antes del Bounds Test.
    ur = v1.integracion(df, [f'lnp_{c}' for c in v1.PRECIOS] + [f'{c}_{s}' for c in v1.PRECIOS for s in ('pos', 'neg')],
                        salida)
    # Misma muestra que el modelo principal del código v1 (2010-05 en adelante; estimación desde 2011-05).
    ad = df.loc['2010-05-01':, [y] + [f'{c}_{s}' for c in v1.PRECIOS for s in ('pos', 'neg')] + v1.CONTROLES].copy()
    ad.index = pd.DatetimeIndex(ad.index, freq='MS')
    S = v1.dummies_estacionales(ad.index)
    filas = [nardl_producto(ad, y, c, S, salida, nsim) for c in v1.PRECIOS]
    t = pd.DataFrame(filas)
    for col in [c for c in t if c.startswith('asimetria_')]:
        t[f'{col}_Holm'] = multipletests(t[col], method='holm')[1]
    t.to_csv(salida / 'nardl_resumen.csv', index=False, encoding='utf-8-sig')
    figura(t, salida, y)
    return t, ur

In [ ]:
if __name__ == '__main__' or 'ipykernel' in sys.modules:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        tabla, integ = main()
    pd.set_option('display.width', 200)
    print('Orden de integración (precios y sumas parciales):')
    print(integ[['variable', 'orden']].to_string(index=False))
    print('\nResumen NARDL por producto:')
    cols = ['producto', 'p', 'n', 'relacion_en_niveles', 'L_pos', 'L_pos_p', 'L_neg', 'L_neg_p',
            'asimetria_largo_HAC_p_Holm', 'asimetria_corto_HAC_p_Holm', 'BG12_p', 'BG24_p']
    print(tabla[cols].round(4).to_string(index=False))